# First-quantization quantum chemistry on a grid

Encode the kinetic-energy operator on a real-space grid. The cpu route builds the Laplacian as a finite-difference matrix; the simulator route emits the block-encoded unitary.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**First-quantization quantum chemistry**: real-space wavefunctions on a grid, with `∇²` as the kinetic operator. The the kinetic-operator kernel emits the encoding circuit directly.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.domains.physics.kernels import grid_laplacian
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

nx, ny = 4, 4
n = nx * ny
dx = dy = 1.0 / (nx + 1)

@trace
def prog(state):
    l_flat = grid_laplacian(nx=nx, ny=ny, nz=1, dx=dx, dy=dy)
    l_mat = shape.reshape(l_flat, result_type=types.tensor("f64", [n, n]), shape=[n, n])
    return apply_linear(l_mat, state)

psi_0 = np.ones(n) / np.sqrt(n)
module = prog(psi_0.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
